## Accessing the Data
Only run this data once to avoid downloading the data multiple times

In [8]:
# import os
# from convokit import Corpus, download

# # Option 1: Allow users to set a custom path via environment variable
# data_dir = os.getenv("BECHDEL_DATA_DIR")  # Users must set this before running

# # Option 2: Fallback to a "./data/movie-corpus" folder in the script's directory
# if data_dir is None:
#     script_dir = os.path.dirname(os.path.abspath(__file__))
#     data_dir = os.path.join(script_dir, "data", "movie-corpus")

# # Ensure the directory exists
# os.makedirs(data_dir, exist_ok=True)

# # Download (or load) the corpus
# corpus = Corpus(filename=download("movie-corpus", data_dir=data_dir, force_local=True))

In [9]:
# corpus.print_summary_stats()

## Converting Speakers to dataframe

In [10]:
import json
import pandas as pd

# Process speaker metadata
data_dir='/Users/brookeye/RISE/Bechdel_Final/Data/1. InputData/convokit/saved-corpora/movie-corpus/'
with open(data_dir + "speakers.json", "r", encoding='utf-8', errors='ignore') as f:
    speaker_data = f.readlines()

speaker_json_str = speaker_data[0] if isinstance(speaker_data, list) and len(speaker_data) > 0 else speaker_data

try:
    speaker_dict = json.loads(speaker_json_str)
except json.JSONDecodeError:
    speaker_dict = json.loads(speaker_json_str.replace("'", '"'))

# Create speakers DataFrame
speakers_records = []
for speaker_id, speaker_info in speaker_dict.items():
    meta = speaker_info['meta']
    speakers_records.append({
        'speaker_id': speaker_id,
        'character_name': meta['character_name'],
        'movie_idx': meta['movie_idx'],
        'movie_name': meta['movie_name'],
        'gender': meta['gender'],
        'credit_pos': meta['credit_pos']
    })

speakers_df = pd.DataFrame(speakers_records)

In [11]:
speakers_df

,speaker_id,character_name,movie_idx,movie_name,gender,credit_pos
0,u0,BIANCA,m0,10 things i hate about you,f,4
1,u2,CAMERON,m0,10 things i hate about you,m,3
2,u3,CHASTITY,m0,10 things i hate about you,?,?
3,u4,JOEY,m0,10 things i hate about you,m,6
4,u5,KAT,m0,10 things i hate about you,f,2
...,...,...,...,...,...,...
9030,u9029,CREALOCK,m616,zulu dawn,?,?
9031,u9033,STUART SMITH,m616,zulu dawn,?,?
9032,u9028,COGHILL,m616,zulu dawn,?,?
9033,u9031,MELVILL,m616,zulu dawn,?,?


## Converting Utterances to dataframe

In [12]:
with open(data_dir + "utterances.json", "r", encoding='utf-8', errors='ignore') as f:
    utterances_data = f.readlines()
# Assuming utterances_data is your list of utterance JSON strings
utterances_records = []

for utterance_str in utterances_data:
    try:
        utterance = json.loads(utterance_str)
        
        utterances_records.append({
            'utterance_id': utterance['id'],
            'conversation_id': utterance['conversation_id'],
            'text': utterance['text'],
            'speaker': utterance['speaker'],
            'movie_id': utterance['meta']['movie_id'],
            'reply_to': utterance['reply-to'],
            'timestamp': utterance['timestamp']
        })
    except json.JSONDecodeError:
        continue

utterances_df = pd.DataFrame(utterances_records)

In [13]:
utterances_df

,utterance_id,conversation_id,text,speaker,movie_id,reply_to,timestamp
0,L1045,L1044,They do not!,u0,m0,L1044,None
1,L1044,L1044,They do to!,u2,m0,None,None
2,L985,L984,I hope so.,u0,m0,L984,None
3,L984,L984,She okay?,u2,m0,None,None
4,L925,L924,Let's go.,u0,m0,L924,None
...,...,...,...,...,...,...,...
304708,L666371,L666369,Lord Chelmsford seems to want me to stay back ...,u9030,m616,L666370,None
304709,L666370,L666369,I'm to take the Sikali with the main column to...,u9034,m616,L666369,None
304710,L666369,L666369,"Your orders, Mr Vereker?",u9030,m616,None,None
304711,L666257,L666256,"Good ones, yes, Mr Vereker. Gentlemen who can ...",u9030,m616,L666256,None


In [14]:
# Merge to get character information with each utterance
combined_df = pd.merge(
    utterances_df,
    speakers_df,
    left_on='speaker',
    right_on='speaker_id',
    how='left'
)

# Display results
combined_df=combined_df.rename(columns={'movie_name': 'title'})
combined_df

,utterance_id,conversation_id,text,speaker,movie_id,reply_to,timestamp,speaker_id,character_name,movie_idx,title,gender,credit_pos
0,L1045,L1044,They do not!,u0,m0,L1044,None,u0,BIANCA,m0,10 things i hate about you,f,4
1,L1044,L1044,They do to!,u2,m0,None,None,u2,CAMERON,m0,10 things i hate about you,m,3
2,L985,L984,I hope so.,u0,m0,L984,None,u0,BIANCA,m0,10 things i hate about you,f,4
3,L984,L984,She okay?,u2,m0,None,None,u2,CAMERON,m0,10 things i hate about you,m,3
4,L925,L924,Let's go.,u0,m0,L924,None,u0,BIANCA,m0,10 things i hate about you,f,4
...,...,...,...,...,...,...,...,...,...,...,...,...,...
304708,L666371,L666369,Lord Chelmsford seems to want me to stay back ...,u9030,m616,L666370,None,u9030,DURNFORD,m616,zulu dawn,?,?
304709,L666370,L666369,I'm to take the Sikali with the main column to...,u9034,m616,L666369,None,u9034,VEREKER,m616,zulu dawn,?,?
304710,L666369,L666369,"Your orders, Mr Vereker?",u9030,m616,None,None,u9030,DURNFORD,m616,zulu dawn,?,?
304711,L666257,L666256,"Good ones, yes, Mr Vereker. Gentlemen who can ...",u9030,m616,L666256,None,u9030,DURNFORD,m616,zulu dawn,?,?


## Merging with Bechdel

In [15]:
bechdel_scores = pd.read_csv('../../Data/1. InputData/bechdel_scores.csv')
bechdel_scores

,year,id,imdbid,title,rating
0,1874,9602,3155794.0,Passage de Venus,0
1,1877,9804,14495706.0,La Rosace Magique,0
2,1878,9603,2221420.0,Sallie Gardner at a Gallop,0
3,1878,9806,12592084.0,Le singe musicien,0
4,1881,9816,7816420.0,Athlete Swinging a Pick,0
...,...,...,...,...,...
10557,2025,11779,29603959.0,Novocaine,1
10558,2025,11785,30840798.0,"Phoenician Scheme, The",3
10559,2025,11786,32550101.0,Straw,3
10560,2025,11787,30253473.0,Materialists,3


In [16]:
# Ensure the 'title' column in bechdel_scores is lowercase
bechdel_scores['title'] = bechdel_scores['title'].str.lower()

# Merge with combined_df (also ensure its 'title' is lowercase for consistency)
combined_df['title'] = combined_df['title'].str.lower()
df = combined_df.merge(bechdel_scores, on='title')
df

,utterance_id,conversation_id,text,speaker,movie_id,reply_to,timestamp,speaker_id,character_name,movie_idx,title,gender,credit_pos,year,id,imdbid,rating
0,L1045,L1044,They do not!,u0,m0,L1044,None,u0,BIANCA,m0,10 things i hate about you,f,4,1999,374,147800.0,3
1,L1044,L1044,They do to!,u2,m0,None,None,u2,CAMERON,m0,10 things i hate about you,m,3,1999,374,147800.0,3
2,L985,L984,I hope so.,u0,m0,L984,None,u0,BIANCA,m0,10 things i hate about you,f,4,1999,374,147800.0,3
3,L984,L984,She okay?,u2,m0,None,None,u2,CAMERON,m0,10 things i hate about you,m,3,1999,374,147800.0,3
4,L925,L924,Let's go.,u0,m0,L924,None,u0,BIANCA,m0,10 things i hate about you,f,4,1999,374,147800.0,3
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
185544,L665991,L665987,"I'm sorry, sir. We only seat by reservation.",u9021,m615,L665990,None,u9021,MAITRE D',m615,young frankenstein,?,?,1974,1077,72431.0,2
185545,L665990,L665987,Food!!,u9023,m615,L665989,None,u9023,MONSTER,m615,young frankenstein,?,?,1974,1077,72431.0,2
185546,L665989,L665987,Do you have a reservation?,u9021,m615,L665988,None,u9021,MAITRE D',m615,young frankenstein,?,?,1974,1077,72431.0,2
185547,L665988,L665987,Food!,u9023,m615,L665987,None,u9023,MONSTER,m615,young frankenstein,?,?,1974,1077,72431.0,2


In [17]:
df['imdbid']=df['imdbid'].astype(int)
df

,utterance_id,conversation_id,text,speaker,movie_id,reply_to,timestamp,speaker_id,character_name,movie_idx,title,gender,credit_pos,year,id,imdbid,rating
0,L1045,L1044,They do not!,u0,m0,L1044,None,u0,BIANCA,m0,10 things i hate about you,f,4,1999,374,147800,3
1,L1044,L1044,They do to!,u2,m0,None,None,u2,CAMERON,m0,10 things i hate about you,m,3,1999,374,147800,3
2,L985,L984,I hope so.,u0,m0,L984,None,u0,BIANCA,m0,10 things i hate about you,f,4,1999,374,147800,3
3,L984,L984,She okay?,u2,m0,None,None,u2,CAMERON,m0,10 things i hate about you,m,3,1999,374,147800,3
4,L925,L924,Let's go.,u0,m0,L924,None,u0,BIANCA,m0,10 things i hate about you,f,4,1999,374,147800,3
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
185544,L665991,L665987,"I'm sorry, sir. We only seat by reservation.",u9021,m615,L665990,None,u9021,MAITRE D',m615,young frankenstein,?,?,1974,1077,72431,2
185545,L665990,L665987,Food!!,u9023,m615,L665989,None,u9023,MONSTER,m615,young frankenstein,?,?,1974,1077,72431,2
185546,L665989,L665987,Do you have a reservation?,u9021,m615,L665988,None,u9021,MAITRE D',m615,young frankenstein,?,?,1974,1077,72431,2
185547,L665988,L665987,Food!,u9023,m615,L665987,None,u9023,MONSTER,m615,young frankenstein,?,?,1974,1077,72431,2


## Add IMDb

In [18]:
title_basics = pd.read_csv('../../Data/1. InputData/title.basics.tsv', sep='\t')
title_basics  = title_basics.rename(columns={'tconst': 'titleId'}).drop(['primaryTitle', 'originalTitle'], axis=1)

title_ratings=pd.read_csv('../../Data/1. InputData/title.ratings.tsv', sep='\t')
title_ratings  = title_ratings.rename(columns={'tconst': 'titleId'})

/var/folders/mm/8lwyn74n6wj3qytmg8w_39bm0000gn/T/ipykernel_38805/3516147833.py:1: DtypeWarning: Columns (4) have mixed types. Specify dtype option on import or set low_memory=False.
  title_basics = pd.read_csv('../../Data/1. InputData/title.basics.tsv', sep='\t')


In [19]:
title_all = pd.read_csv('../../Data/1. InputData/title.akas.tsv', sep='\t')
title_all=title_all.drop(['region', 'types', 'attributes'], axis =1)
title_all = title_all[title_all['isOriginalTitle'] == 1]
title_all['title'] = title_all['title'].str.lower()
title_all = title_all.drop_duplicates('title', keep='first')
title_all=title_all.merge(title_ratings, on='titleId').merge(title_basics, on='titleId')

oscar_winning = pd.read_csv('../../Data/1. InputData/oscars_df.csv')
oscar_winning=oscar_winning.drop(['Unnamed: 0', 'Oscar Year'], axis=1)
oscar_winning=oscar_winning.rename(columns={'Film' : 'title'})
oscar_winning = set(oscar_winning['title'].str.lower())
title_all['oscar'] = title_all['title'].str.lower().apply(lambda x: 1 if x in oscar_winning else 0)
title_all=title_all.drop(['isOriginalTitle', 'language', 'ordering'], axis=1)

In [20]:
def remove_leading_zeros(s):
    return s.lstrip('0') or '0'

title_all['titleId']=title_all['titleId'].str.replace('t', '')
title_all['titleId']= [remove_leading_zeros(num) for num in title_all['titleId']]
title_all['titleId']= title_all['titleId'].astype(int)
title_all=title_all.rename(columns={'titleId' : 'imdbid'})
title_all

,imdbid,title,averageRating,numVotes,titleType,isAdult,startYear,endYear,runtimeMinutes,genres,oscar
0,1,carmencita,5.7,2165,short,0,1894,\N,1,"Documentary,Short",0
1,2,le clown et ses chiens,5.5,296,short,0,1892,\N,5,"Animation,Short",0
2,3,pauvre pierrot,6.5,2224,short,0,1892,\N,5,"Animation,Comedy,Romance",0
3,4,un bon bock,5.3,190,short,0,1892,\N,12,"Animation,Short",0
4,5,blacksmith scene,6.2,2963,short,0,1893,\N,1,Short,0
...,...,...,...,...,...,...,...,...,...,...,...
1104513,9916708,horrid henry goes gross,7.7,10,tvEpisode,0,2012,\N,\N,"Adventure,Animation,Comedy",0
1104514,9916724,hay que ser paciente,6.6,5,short,0,2015,\N,3,"Documentary,Short",0
1104515,9916730,6 gunn,7.0,13,movie,0,2017,\N,116,Drama,0
1104516,9916840,horrid henry's comic caper,6.8,12,tvEpisode,0,2014,\N,11,"Adventure,Animation,Comedy",0


In [21]:
df_with_imdb=df.merge(title_all,on='imdbid')
df_with_imdb

,utterance_id,conversation_id,text,speaker,movie_id,reply_to,timestamp,speaker_id,character_name,movie_idx,...,title_y,averageRating,numVotes,titleType,isAdult,startYear,endYear,runtimeMinutes,genres,oscar
0,L1045,L1044,They do not!,u0,m0,L1044,None,u0,BIANCA,m0,...,10 things i hate about you,7.4,424659,movie,0,1999,\N,97,"Comedy,Drama,Romance",0
1,L1044,L1044,They do to!,u2,m0,None,None,u2,CAMERON,m0,...,10 things i hate about you,7.4,424659,movie,0,1999,\N,97,"Comedy,Drama,Romance",0
2,L985,L984,I hope so.,u0,m0,L984,None,u0,BIANCA,m0,...,10 things i hate about you,7.4,424659,movie,0,1999,\N,97,"Comedy,Drama,Romance",0
3,L984,L984,She okay?,u2,m0,None,None,u2,CAMERON,m0,...,10 things i hate about you,7.4,424659,movie,0,1999,\N,97,"Comedy,Drama,Romance",0
4,L925,L924,Let's go.,u0,m0,L924,None,u0,BIANCA,m0,...,10 things i hate about you,7.4,424659,movie,0,1999,\N,97,"Comedy,Drama,Romance",0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
149464,L665991,L665987,"I'm sorry, sir. We only seat by reservation.",u9021,m615,L665990,None,u9021,MAITRE D',m615,...,young frankenstein,8.0,176404,movie,0,1974,\N,106,Comedy,0
149465,L665990,L665987,Food!!,u9023,m615,L665989,None,u9023,MONSTER,m615,...,young frankenstein,8.0,176404,movie,0,1974,\N,106,Comedy,0
149466,L665989,L665987,Do you have a reservation?,u9021,m615,L665988,None,u9021,MAITRE D',m615,...,young frankenstein,8.0,176404,movie,0,1974,\N,106,Comedy,0
149467,L665988,L665987,Food!,u9023,m615,L665987,None,u9023,MONSTER,m615,...,young frankenstein,8.0,176404,movie,0,1974,\N,106,Comedy,0


In [22]:
df_with_imdb.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 149469 entries, 0 to 149468
Data columns (total 27 columns):
 #   Column           Non-Null Count   Dtype  
---  ------           --------------   -----  
 0   utterance_id     149469 non-null  object 
 1   conversation_id  149469 non-null  object 
 2   text             149469 non-null  object 
 3   speaker          149469 non-null  object 
 4   movie_id         149469 non-null  object 
 5   reply_to         108481 non-null  object 
 6   timestamp        0 non-null       object 
 7   speaker_id       149469 non-null  object 
 8   character_name   149469 non-null  object 
 9   movie_idx        149469 non-null  object 
 10  title_x          149469 non-null  object 
 11  gender           149469 non-null  object 
 12  credit_pos       149469 non-null  object 
 13  year             149469 non-null  int64  
 14  id               149469 non-null  int64  
 15  imdbid           149469 non-null  int64  
 16  rating           149469 non-null  int6

In [23]:
df_cleaned=df_with_imdb.drop(['timestamp', 'title_y', 'startYear', 'endYear', 'isAdult', 'titleType'], axis=1)

In [24]:
# df_cleaned=df_cleaned.drop(['movie_idy'], axis=1)
df_cleaned=df_cleaned.rename(columns={'movie_idx' : 'movie_id', 'title_x': 'title', 'rating': 'bechdel_score', 'averageRating': 'imdb_score'})
df_cleaned['gender']=df_cleaned['gender'].str.lower()

## Resolving Gender==? issue

In [25]:
unknown_gender = df_with_imdb[df_with_imdb['gender'] == '?']

In [26]:
print(unknown_gender['character_name'])

70         CHASTITY
71         CHASTITY
73         CHASTITY
75         CHASTITY
77         CHASTITY
            ...    
149464    MAITRE D'
149465      MONSTER
149466    MAITRE D'
149467      MONSTER
149468    MAITRE D'
Name: character_name, Length: 29969, dtype: object


In [27]:
unknown_gender['character_name']=unknown_gender['character_name'].str.lower()

/var/folders/mm/8lwyn74n6wj3qytmg8w_39bm0000gn/T/ipykernel_38805/1493671143.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  unknown_gender['character_name']=unknown_gender['character_name'].str.lower()


In [28]:
unknown_gender

,utterance_id,conversation_id,text,speaker,movie_id,reply_to,timestamp,speaker_id,character_name,movie_idx,...,title_y,averageRating,numVotes,titleType,isAdult,startYear,endYear,runtimeMinutes,genres,oscar
70,L952,L952,You think you ' re the only sophomore at the p...,u3,m0,None,None,u3,chastity,m0,...,10 things i hate about you,7.4,424659,movie,0,1999,\N,97,"Comedy,Drama,Romance",0
71,L660,L659,I don't have to be home 'til two.,u3,m0,L659,None,u3,chastity,m0,...,10 things i hate about you,7.4,424659,movie,0,1999,\N,97,"Comedy,Drama,Romance",0
73,L600,L598,All I know is -- I'd give up my private line t...,u3,m0,L599,None,u3,chastity,m0,...,10 things i hate about you,7.4,424659,movie,0,1999,\N,97,"Comedy,Drama,Romance",0
75,L598,L598,"Bianca, I don't think the highlights of dating...",u3,m0,None,None,u3,chastity,m0,...,10 things i hate about you,7.4,424659,movie,0,1999,\N,97,"Comedy,Drama,Romance",0
77,L596,L595,Is he oily or dry?,u3,m0,L595,None,u3,chastity,m0,...,10 things i hate about you,7.4,424659,movie,0,1999,\N,97,"Comedy,Drama,Romance",0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
149464,L665991,L665987,"I'm sorry, sir. We only seat by reservation.",u9021,m615,L665990,None,u9021,maitre d',m615,...,young frankenstein,8.0,176404,movie,0,1974,\N,106,Comedy,0
149465,L665990,L665987,Food!!,u9023,m615,L665989,None,u9023,monster,m615,...,young frankenstein,8.0,176404,movie,0,1974,\N,106,Comedy,0
149466,L665989,L665987,Do you have a reservation?,u9021,m615,L665988,None,u9021,maitre d',m615,...,young frankenstein,8.0,176404,movie,0,1974,\N,106,Comedy,0
149467,L665988,L665987,Food!,u9023,m615,L665987,None,u9023,monster,m615,...,young frankenstein,8.0,176404,movie,0,1974,\N,106,Comedy,0


In [31]:
import gender_guesser.detector as gender #if encountering an error, run "pip install gender-guesser" in terminal
import pandas as pd

d = gender.Detector()

def get_gender(name):
    # Handle empty/missing names
    if pd.isna(name) or not str(name).strip():
        return '?'
    
    name = str(name).strip().lower()
    
    # Extended lists of gendered words
    male_indicators = {'mr', 'mister', 'sir', 'father', 'uncle', 'dad', 'brother', 'lord',
                      'boy', 'man', 'male', 'gentleman', 'guy', 'groom', 'dennis\'', 'stilgar', 'kynes', 'thufir', 'michaels\'', 'bruce\'s', 'billy\'s', 'monster'}
    female_indicators = {'mrs', 'miss', 'ms', 'lady', 'aunt', 'mother', 'mom', 
                        'sister', 'woman', 'lady', 'girl', 'female', 'gal', 'wife', "ma", 'clementine\'s', 'bride', 'sexy', 'reese', 'waitress', 'rose\'s'}
    
    # Check ALL words in the name for gender indicators
    for word in name.split():
        # Check for female indicators first (more specific titles)
        if any(word.startswith(ind) for ind in female_indicators):
            return 'f'
        # Then check male indicators
        if any(word.startswith(ind) for ind in male_indicators):
            return 'm'
    
    # Try all possible first names (from right to left)
    # This catches cases like "nice guy eddie" where the first name is last
    for possible_first_name in reversed(name.split()):
        guess = d.get_gender(possible_first_name.capitalize())
        if guess in ['male', 'mostly_male']:
            return 'm'
        if guess in ['female', 'mostly_female']:
            return 'f'
    
    # Final check for obvious gendered words anywhere in the name
    if any(word in name for word in female_indicators):
        return 'f'
    if any(word in name for word in male_indicators):
        return 'm'
    
    return '?'  # Default for unknown cases

# Apply to your dataframe
unknown_gender['character_name'] = unknown_gender['character_name'].str.lower()
unknown_gender['gender'] = unknown_gender['character_name'].apply(get_gender)

# Show results
unknown_gender

/var/folders/mm/8lwyn74n6wj3qytmg8w_39bm0000gn/T/ipykernel_38805/1116833760.py:46: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  unknown_gender['character_name'] = unknown_gender['character_name'].str.lower()
/var/folders/mm/8lwyn74n6wj3qytmg8w_39bm0000gn/T/ipykernel_38805/1116833760.py:47: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  unknown_gender['gender'] = unknown_gender['character_name'].apply(get_gender)


,utterance_id,conversation_id,text,speaker,movie_id,reply_to,timestamp,speaker_id,character_name,movie_idx,...,title_y,averageRating,numVotes,titleType,isAdult,startYear,endYear,runtimeMinutes,genres,oscar
70,L952,L952,You think you ' re the only sophomore at the p...,u3,m0,None,None,u3,chastity,m0,...,10 things i hate about you,7.4,424659,movie,0,1999,\N,97,"Comedy,Drama,Romance",0
71,L660,L659,I don't have to be home 'til two.,u3,m0,L659,None,u3,chastity,m0,...,10 things i hate about you,7.4,424659,movie,0,1999,\N,97,"Comedy,Drama,Romance",0
73,L600,L598,All I know is -- I'd give up my private line t...,u3,m0,L599,None,u3,chastity,m0,...,10 things i hate about you,7.4,424659,movie,0,1999,\N,97,"Comedy,Drama,Romance",0
75,L598,L598,"Bianca, I don't think the highlights of dating...",u3,m0,None,None,u3,chastity,m0,...,10 things i hate about you,7.4,424659,movie,0,1999,\N,97,"Comedy,Drama,Romance",0
77,L596,L595,Is he oily or dry?,u3,m0,L595,None,u3,chastity,m0,...,10 things i hate about you,7.4,424659,movie,0,1999,\N,97,"Comedy,Drama,Romance",0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
149464,L665991,L665987,"I'm sorry, sir. We only seat by reservation.",u9021,m615,L665990,None,u9021,maitre d',m615,...,young frankenstein,8.0,176404,movie,0,1974,\N,106,Comedy,0
149465,L665990,L665987,Food!!,u9023,m615,L665989,None,u9023,monster,m615,...,young frankenstein,8.0,176404,movie,0,1974,\N,106,Comedy,0
149466,L665989,L665987,Do you have a reservation?,u9021,m615,L665988,None,u9021,maitre d',m615,...,young frankenstein,8.0,176404,movie,0,1974,\N,106,Comedy,0
149467,L665988,L665987,Food!,u9023,m615,L665987,None,u9023,monster,m615,...,young frankenstein,8.0,176404,movie,0,1974,\N,106,Comedy,0


In [32]:
unknown_final = unknown_gender[unknown_gender['gender'] == '?']

In [33]:
import gender_guesser.detector as gender
import pandas as pd

d = gender.Detector()

def get_gender(name):
    if pd.isna(name) or not str(name).strip():
        return '?'
    name = str(name).strip().lower()
    male_indicators = {'mr', 'mister', 'sir', 'father', 'uncle', 'dad', 'brother', 'lord',
                      'boy', 'man', 'male', 'gentleman', 'guy', 'groom', 'dennis\'', 'stilgar', 'kynes', 'thufir', 'michaels\'', 'bruce\'s', 'billy\'s', 'monster'}
    female_indicators = {'mrs', 'miss', 'ms', 'lady', 'aunt', 'mother', 'mom', 
                        'sister', 'woman', 'lady', 'girl', 'female', 'gal', 'wife', "ma", 'clementine\'s', 'bride', 'sexy', 'reese', 'waitress', 'rose\'s'}
    for word in name.split():
        if any(word.startswith(ind) for ind in female_indicators):
            return 'f'
        if any(word.startswith(ind) for ind in male_indicators):
            return 'm'
    for possible_first_name in reversed(name.split()):
        guess = d.get_gender(possible_first_name.capitalize())
        if guess in ['male', 'mostly_male']:
            return 'm'
        if guess in ['female', 'mostly_female']:
            return 'f'
    if any(word in name for word in female_indicators):
        return 'f'
    if any(word in name for word in male_indicators):
        return 'm'
    return '?'  # Default for unknown cases

# Filter unknowns, lowercase names, predict gender
unknown_gender = df_cleaned[df_cleaned['gender'] == '?'].copy()
unknown_gender['character_name'] = unknown_gender['character_name'].str.lower()
unknown_gender['gender'] = unknown_gender['character_name'].apply(get_gender)

# Update original dataframe with new gender assignments
df_cleaned.loc[unknown_gender.index, 'gender'] = unknown_gender['gender']

In [34]:
df_cleaned=df_cleaned[df_cleaned['gender']!='?']

In [35]:
df_cleaned

,utterance_id,conversation_id,text,speaker,movie_id,reply_to,speaker_id,character_name,movie_id,title,...,credit_pos,year,id,imdbid,bechdel_score,imdb_score,numVotes,runtimeMinutes,genres,oscar
0,L1045,L1044,They do not!,u0,m0,L1044,u0,BIANCA,m0,10 things i hate about you,...,4,1999,374,147800,3,7.4,424659,97,"Comedy,Drama,Romance",0
1,L1044,L1044,They do to!,u2,m0,None,u2,CAMERON,m0,10 things i hate about you,...,3,1999,374,147800,3,7.4,424659,97,"Comedy,Drama,Romance",0
2,L985,L984,I hope so.,u0,m0,L984,u0,BIANCA,m0,10 things i hate about you,...,4,1999,374,147800,3,7.4,424659,97,"Comedy,Drama,Romance",0
3,L984,L984,She okay?,u2,m0,None,u2,CAMERON,m0,10 things i hate about you,...,3,1999,374,147800,3,7.4,424659,97,"Comedy,Drama,Romance",0
4,L925,L924,Let's go.,u0,m0,L924,u0,BIANCA,m0,10 things i hate about you,...,4,1999,374,147800,3,7.4,424659,97,"Comedy,Drama,Romance",0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
149464,L665991,L665987,"I'm sorry, sir. We only seat by reservation.",u9021,m615,L665990,u9021,MAITRE D',m615,young frankenstein,...,?,1974,1077,72431,2,8.0,176404,106,Comedy,0
149465,L665990,L665987,Food!!,u9023,m615,L665989,u9023,MONSTER,m615,young frankenstein,...,?,1974,1077,72431,2,8.0,176404,106,Comedy,0
149466,L665989,L665987,Do you have a reservation?,u9021,m615,L665988,u9021,MAITRE D',m615,young frankenstein,...,?,1974,1077,72431,2,8.0,176404,106,Comedy,0
149467,L665988,L665987,Food!,u9023,m615,L665987,u9023,MONSTER,m615,young frankenstein,...,?,1974,1077,72431,2,8.0,176404,106,Comedy,0


## Final Edits

In [ ]:
df_cleaned = df_cleaned[df_cleaned["text"] != '']
df_cleaned.to_csv('df_clean_original.csv', index=False)

In [40]:
# Create the first DataFrame: hard (score == 3)
df_binarized_hard = df_cleaned.copy()
df_binarized_hard['bechdel_score'] = df_binarized_hard['bechdel_score'].apply(lambda x: 1 if x == 3 else 0)

# Create the second DataFrame: medium (score >= 2)
df_binarized_medium = df_cleaned.copy()
df_binarized_medium['bechdel_score'] = df_binarized_medium['bechdel_score'].apply(lambda x: 1 if x >= 2 else 0)

# Create the third DataFrame: easy (score >= 1)
df_binarized_easy = df_cleaned.copy()
df_binarized_easy['bechdel_score'] = df_binarized_easy['bechdel_score'].apply(lambda x: 1 if x >= 1 else 0)

# Save the DataFrames to CSV files
df_binarized_hard.to_csv('df_binarized_hard.csv', index=False)
df_binarized_medium.to_csv('df_binarized_medium.csv', index=False)
df_binarized_easy.to_csv('df_binarized_easy.csv', index=False)